# Survived データセットの探索と分類モデルの訓練

このノートブックでは、Titanic 生存予測データセットを使った分類問題を TDD で実装します。

## 1. 環境設定とパスの確認

In [6]:
import java.io.File

// 現在のワーキングディレクトリを確認
val currentDir = File(".").absolutePath
println("現在のディレクトリ: $currentDir")

// データファイルのパスを自動検出
val possiblePaths = listOf(
    "src/main/resources/data/Survived.csv",           // app/kotlin から実行
    "../src/main/resources/data/Survived.csv",        // notebook から実行
    "app/kotlin/src/main/resources/data/Survived.csv", // プロジェクトルートから実行
    "../../src/main/resources/data/Survived.csv"      // さらに深い場所から実行
)

val dataPath = possiblePaths.firstOrNull { File(it).exists() }
    ?: error("Survived.csv が見つかりません。現在のディレクトリ: $currentDir")

println("データファイル: $dataPath")
println("ファイル存在確認: ${File(dataPath).exists()}")

現在のディレクトリ: C:\Users\PC202411-1\IdeaProjects\case-study-game-dev\app\kotlin\notebook\.
データファイル: ../src/main/resources/data/Survived.csv
ファイル存在確認: true


## 2. データの読み込みと概要確認

In [7]:
// Smile ML ライブラリの依存関係を追加
@file:Repository("https://jitpack.io")
@file:DependsOn("com.github.haifengl:smile-core:3.0.2")
@file:DependsOn("com.github.haifengl:smile-kotlin:3.0.2")
@file:DependsOn("org.jetbrains.kotlinx:dataframe:0.13.1")

println("依存関係の設定完了")

依存関係の設定完了


In [8]:
import smile.classification.DecisionTree
import smile.data.DataFrame
import smile.data.formula.Formula
import smile.data.vector.DoubleVector
import smile.data.vector.IntVector
import java.io.Serializable
import java.util.Properties

/**
 * Survived データセットを分類する決定木モデル
 * Titanic の生存予測を行う
 */
class SurvivedClassifier(val maxDepth: Int = 9) : Serializable {
    
    var model: DecisionTree? = null
        private set
    
    init {
        require(maxDepth >= 1) { "maxDepth must be at least 1" }
    }
    
    /**
     * CSV ファイルからデータを読み込む
     * 欠損値は Pclass ごとの平均値で補完される
     * Sex 列は male ダミー変数にエンコードされる (male=1, female=0)
     */
    fun loadData(filePath: String): Pair<Array<DoubleArray>, IntArray> {
        val file = java.io.File(filePath)
        require(file.exists()) { "File not found: $filePath" }
        
        val lines = file.readLines()
        require(lines.isNotEmpty()) { "Empty file: $filePath" }
        
        // ヘッダー行を解析（BOMを除去）
        val headerLine = lines[0].replace("\uFEFF", "").trim()
        val header = headerLine.split(",").map { it.trim() }
        val pclassIdx = header.indexOf("Pclass")
        val ageIdx = header.indexOf("Age")
        val sexIdx = header.indexOf("Sex")
        val survivedIdx = header.indexOf("Survived")
        
        require(pclassIdx >= 0 && ageIdx >= 0 && sexIdx >= 0 && survivedIdx >= 0) {
            "Required columns not found in CSV"
        }
        
        // 第1パス: Pclass ごとの Age 平均値を計算
        val ageByPclass = mutableMapOf<Int, MutableList<Double>>()
        
        for (i in 1 until lines.size) {
            val values = lines[i].split(",")
            if (values.size != header.size) continue
            
            val pclass = values[pclassIdx].trim().toIntOrNull() ?: continue
            val age = values[ageIdx].trim().toDoubleOrNull() ?: continue
            
            ageByPclass.getOrPut(pclass) { mutableListOf() }.add(age)
        }
        
        // Pclass ごとの Age 平均値
        val ageMeanByPclass = ageByPclass.mapValues { (_, ages) ->
            if (ages.isNotEmpty()) ages.average() else 0.0
        }
        
        // 第2パス: データ行を読み込み（欠損値を補完）
        val validRows = mutableListOf<Pair<DoubleArray, Int>>()
        
        for (i in 1 until lines.size) {
            val values = lines[i].split(",")
            if (values.size != header.size) continue
            
            try {
                val pclass = values[pclassIdx].trim().toIntOrNull() ?: continue
                val survived = values[survivedIdx].trim().toIntOrNull() ?: continue
                
                // Age は欠損値の場合、Pclass ごとの平均値で補完
                val age = values[ageIdx].trim().toDoubleOrNull()
                    ?: ageMeanByPclass[pclass]
                    ?: continue
                
                // Sex を male ダミー変数にエンコード (male=1, female=0)
                val sex = values[sexIdx].trim().lowercase()
                val male = if (sex == "male") 1.0 else 0.0
                
                val features = doubleArrayOf(pclass.toDouble(), age, male)
                validRows.add(Pair(features, survived))
            } catch (e: NumberFormatException) {
                continue
            }
        }
        
        require(validRows.isNotEmpty()) { "No valid data found in CSV" }
        
        val X = validRows.map { it.first }.toTypedArray()
        val y = validRows.map { it.second }.toIntArray()
        
        return Pair(X, y)
    }
    
    /**
     * モデルを訓練する
     */
    fun train(X: Array<DoubleArray>, y: IntArray) {
        require(X.isNotEmpty() && y.isNotEmpty()) { "Training data cannot be empty" }
        require(X.size == y.size) {
            "X and y must have the same length: ${X.size} != ${y.size}"
        }
        
        // DataFrame を作成
        val data = DataFrame.of(
            DoubleVector.of("Pclass", X.map { it[0] }.toDoubleArray()),
            DoubleVector.of("Age", X.map { it[1] }.toDoubleArray()),
            DoubleVector.of("male", X.map { it[2] }.toDoubleArray()),
            IntVector.of("Survived", y)
        )
        
        // モデルの訓練
        val formula = Formula.lhs("Survived")
        val props = Properties()
        props.setProperty("smile.decision_tree.max_depth", maxDepth.toString())
        model = DecisionTree.fit(formula, data, props)
    }
    
    /**
     * 予測を実行する
     */
    fun predict(X: Array<DoubleArray>): IntArray {
        requireNotNull(model) { "Model has not been trained yet" }
        
        val dummySurvived = IntArray(X.size) { 0 }
        val testData = DataFrame.of(
            DoubleVector.of("Pclass", X.map { it[0] }.toDoubleArray()),
            DoubleVector.of("Age", X.map { it[1] }.toDoubleArray()),
            DoubleVector.of("male", X.map { it[2] }.toDoubleArray()),
            IntVector.of("Survived", dummySurvived)
        )
        
        return model!!.predict(testData)
    }
    
    /**
     * モデルの性能を評価する
     */
    fun evaluate(X: Array<DoubleArray>, y: IntArray): Double {
        requireNotNull(model) { "Model has not been trained yet" }
        
        val predictions = predict(X)
        val correct = predictions.zip(y.toTypedArray()).count { (pred, actual) -> pred == actual }
        return correct.toDouble() / y.size
    }
    
    /**
     * 訓練済みモデルをファイルに保存する
     */
    fun saveModel(filePath: String) {
        requireNotNull(model) { "No trained model to save" }
        
        java.io.ObjectOutputStream(java.io.FileOutputStream(filePath)).use { oos ->
            oos.writeObject(model)
        }
    }
    
    /**
     * 保存されたモデルをファイルから読み込む
     */
    fun loadModel(filePath: String) {
        java.io.ObjectInputStream(java.io.FileInputStream(filePath)).use { ois ->
            @Suppress("UNCHECKED_CAST")
            model = ois.readObject() as DecisionTree
        }
    }
}

// モデルの作成
val classifier = SurvivedClassifier(maxDepth = 9)

// データの読み込み
val (X, y) = classifier.loadData(dataPath)

println("=".repeat(60))
println("データの概要")
println("=".repeat(60))
println("サンプル数: ${X.size}")
println("特徴量数: ${X[0].size}")
println("クラス数: 2 (0: 死亡, 1: 生存)")
println()

// データの最初の5行を表示
println("最初の5サンプル:")
println("Pclass, Age, male, Survived")
for (i in 0 until minOf(5, X.size)) {
    println("${X[i][0]}, ${X[i][1]}, ${X[i][2]}, ${y[i]}")
}

データの概要
サンプル数: 891
特徴量数: 3
クラス数: 2 (0: 死亡, 1: 生存)

最初の5サンプル:
Pclass, Age, male, Survived
3.0, 22.0, 1.0, 0
1.0, 38.0, 0.0, 1
3.0, 26.0, 0.0, 1
1.0, 35.0, 0.0, 1
3.0, 35.0, 1.0, 0


## 3. Lets-Plot の初期化

In [9]:
%use lets-plot

## 4. データの可視化

### 4.1 特徴量の分布（ヒストグラム）- Age

In [10]:
// Age の分布をヒストグラムで可視化
val ageValues = X.map { it[1] }

val p = letsPlot() + 
    geomHistogram(bins = 20) { x = ageValues }

p

0 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 
 
 70 
 
 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 150 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 
 
 count 
 
 
 
 
 x

### 4.2 Pclass の分布（棒グラフ）

In [11]:
// Pclass の分布を棒グラフで可視化
val pclassValues = X.map { it[0].toInt() }
val pclassCounts = pclassValues.groupBy { it }.mapValues { it.value.size }

val pclassKeys = pclassCounts.keys.toList()
val pclassCountsValues = pclassCounts.values.toList()

val p = letsPlot() + 
    geomBar(stat = Stat.identity, alpha = 0.8) { 
        x = pclassKeys
        y = pclassCountsValues
    }

p

0.5 
 
 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 
 
 1.5 
 
 
 
 
 
 
 
 
 2.0 
 
 
 
 
 
 
 
 
 2.5 
 
 
 
 
 
 
 
 
 3.0 
 
 
 
 
 
 
 
 
 3.5 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 400 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x

### 4.3 散布図 - Age vs Survived

In [12]:
// Age vs Survived の散布図
val ageData = X.map { it[1] }
val survivedData = y.toList()

val p = letsPlot() + 
    geomPoint(size = 3.0, alpha = 0.5) { 
        x = ageData
        y = survivedData
    }

p

0 
 
 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 
 
 70 
 
 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 
 
 
 
 0.0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1.0 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x

### 4.4 散布図 - Pclass vs Age (色: Survived)

In [13]:
// Pclass vs Age の散布図（Survived で色分け）
val pclassData = X.map { it[0] }
val survivedColors = y.map { it.toString() }

val p = letsPlot() + 
    geomPoint(size = 3.0, alpha = 0.5) { 
        x = pclassData
        y = ageData
        color = survivedColors
    }

p

1.0 
 
 
 
 
 
 
 
 
 1.5 
 
 
 
 
 
 
 
 
 2.0 
 
 
 
 
 
 
 
 
 2.5 
 
 
 
 
 
 
 
 
 3.0 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 70 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x 
 
 
 
 
 
 
 
 
 color 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 1

## 5. 基本統計量の確認

In [14]:
val featureNames = listOf("Pclass", "Age", "male")

println("基本統計量:")
println("-".repeat(60))

featureNames.forEachIndexed { idx, name ->
    val values = X.map { it[idx] }
    val min = values.minOrNull() ?: 0.0
    val max = values.maxOrNull() ?: 0.0
    val mean = values.average()
    val sorted = values.sorted()
    val median = if (sorted.size % 2 == 0) {
        (sorted[sorted.size / 2 - 1] + sorted[sorted.size / 2]) / 2.0
    } else {
        sorted[sorted.size / 2]
    }
    
    println("$name:")
    println("  最小値: %.2f".format(min))
    println("  最大値: %.2f".format(max))
    println("  平均値: %.2f".format(mean))
    println("  中央値: %.2f".format(median))
    println()
}

基本統計量:
------------------------------------------------------------
Pclass:
  最小値: 1.00
  最大値: 3.00
  平均値: 2.31
  中央値: 3.00

Age:
  最小値: 0.42
  最大値: 80.00
  平均値: 29.29
  中央値: 26.00

male:
  最小値: 0.00
  最大値: 1.00
  平均値: 0.65
  中央値: 1.00



## 6. クラスの分布

In [15]:
println("クラスの分布:")
println("-".repeat(60))

val total = y.size.toDouble()
val died = y.count { it == 0 }
val survived = y.count { it == 1 }

println("0 (死亡): $died 件 (%.1f%%)".format(died / total * 100))
println("1 (生存): $survived 件 (%.1f%%)".format(survived / total * 100))
println()

クラスの分布:
------------------------------------------------------------
0 (死亡): 549 件 (61.6%)
1 (生存): 342 件 (38.4%)



### クラス分布の棒グラフ

In [16]:
// クラス分布の棒グラフ
val classNames = listOf("死亡", "生存")
val classCounts = listOf(died, survived)

val p = letsPlot() + 
    geomBar(stat = Stat.identity, alpha = 0.8) { 
        x = classNames
        y = classCounts
    }

p

死亡 
 
 
 
 
 
 
 
 
 生存 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 400 
 
 
 
 
 
 
 500 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x

## 7. モデルの訓練

In [17]:
println("モデルの訓練中...")
val startTime = System.currentTimeMillis()
classifier.train(X, y)
val trainingTime = System.currentTimeMillis() - startTime

println("訓練完了！")
println("訓練時間: ${trainingTime}ms")
println()

モデルの訓練中...
訓練完了！
訓練時間: 78ms



## 8. モデルの評価

In [18]:
val trainAccuracy = classifier.evaluate(X, y)
println("=".repeat(60))
println("モデルの評価結果")
println("=".repeat(60))
println("訓練正解率: %.2f%%".format(trainAccuracy * 100))
println()

モデルの評価結果
訓練正解率: 70.59%



## 9. 混同行列

In [19]:
println("混同行列:")
val predictions = classifier.predict(X)
val classes = listOf(0, 1)
val classNamesMap = mapOf(0 to "死亡", 1 to "生存")

println("実際 \\ 予測 |   0 |   1")
println("-".repeat(30))

// 混同行列のデータを準備
val confusionMatrix = mutableListOf<Triple<String, String, Int>>()

classes.forEach { actualClass ->
    val actualIndices = y.indices.filter { y[it] == actualClass }
    val className = classNamesMap[actualClass]!!
    print("%-10s | ".format(className))
    classes.forEach { predClass ->
        val count = actualIndices.count { predictions[it] == predClass }
        print("%3d | ".format(count))
        confusionMatrix.add(Triple(className, classNamesMap[predClass]!!, count))
    }
    println()
}
println()

混同行列:
実際 \ 予測 |   0 |   1
------------------------------
死亡         | 468 |  81 | 
生存         | 181 | 161 | 



### 混同行列のヒートマップ

In [20]:
// 混同行列のヒートマップ
val actualLabels = confusionMatrix.map { it.first }
val predictedLabels = confusionMatrix.map { it.second }
val countValues = confusionMatrix.map { it.third }

val p = letsPlot() + 
    geomTile(alpha = 0.9) { 
        x = predictedLabels
        y = actualLabels
        fill = countValues
    }

p

死亡 
 
 
 
 
 
 
 
 
 生存 
 
 
 
 
 
 
 
 
 
 
 死亡 
 
 
 
 
 
 
 生存 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x 
 
 
 
 
 
 
 
 
 fill 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 100 
 
 
 
 
 
 
 
 
 200 
 
 
 
 
 
 
 
 
 300 
 
 
 
 
 
 
 
 
 400

### 精度指標の計算

In [21]:
println("精度指標:")
println("-".repeat(60))

val tp = predictions.indices.count { predictions[it] == 1 && y[it] == 1 }  // True Positive
val tn = predictions.indices.count { predictions[it] == 0 && y[it] == 0 }  // True Negative
val fp = predictions.indices.count { predictions[it] == 1 && y[it] == 0 }  // False Positive
val fn = predictions.indices.count { predictions[it] == 0 && y[it] == 1 }  // False Negative

val precision = if (tp + fp > 0) tp.toDouble() / (tp + fp) else 0.0
val recall = if (tp + fn > 0) tp.toDouble() / (tp + fn) else 0.0
val f1Score = if (precision + recall > 0) 2 * precision * recall / (precision + recall) else 0.0

println("Precision (精度): %.2f%%".format(precision * 100))
println("Recall (再現率): %.2f%%".format(recall * 100))
println("F1 Score: %.2f%%".format(f1Score * 100))
println()

精度指標:
------------------------------------------------------------
Precision (精度): 66.53%
Recall (再現率): 47.08%
F1 Score: 55.14%



## 10. クラス別の性能

In [22]:
println("クラス別の性能:")
println("-".repeat(60))

val classAccuracies = mutableListOf<Pair<String, Double>>()

classes.forEach { targetClass ->
    val indices = y.indices.filter { y[it] == targetClass }
    val correct = indices.count { predictions[it] == targetClass }
    val total = indices.size
    val classAccuracy = if (total > 0) correct.toDouble() / total else 0.0
    classAccuracies.add(Pair(classNamesMap[targetClass]!!, classAccuracy))

    println("${classNamesMap[targetClass]} ($targetClass):")
    println("  正解率: %.2f%% ($correct / $total)".format(classAccuracy * 100))
}
println()

クラス別の性能:
------------------------------------------------------------
死亡 (0):
  正解率: 85.25% (468 / 549)
生存 (1):
  正解率: 47.08% (161 / 342)



### クラス別正解率の棒グラフ

In [23]:
// クラス別正解率の棒グラフ
val classNamesAcc = classAccuracies.map { it.first }
val accuracyValues = classAccuracies.map { it.second * 100 }

val p = letsPlot() + 
    geomBar(stat = Stat.identity, alpha = 0.8) { 
        x = classNamesAcc
        y = accuracyValues
    }

p

死亡 
 
 
 
 
 
 
 
 
 生存 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 30 
 
 
 
 
 
 
 40 
 
 
 
 
 
 
 50 
 
 
 
 
 
 
 60 
 
 
 
 
 
 
 70 
 
 
 
 
 
 
 80 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x

## 11. 個別予測の例

In [24]:
println("個別予測の例:")
println("-".repeat(60))

val testSamples = listOf(
    Triple(doubleArrayOf(1.0, 35.0, 0.0), 1, "1等客室、35歳、女性"),
    Triple(doubleArrayOf(3.0, 25.0, 1.0), 0, "3等客室、25歳、男性"),
    Triple(doubleArrayOf(2.0, 28.0, 0.0), 1, "2等客室、28歳、女性")
)

testSamples.forEachIndexed { i, (sample, expected, description) ->
    val prediction = classifier.predict(arrayOf(sample))[0]
    val isCorrect = prediction == expected
    val mark = if (isCorrect) "✓" else "✗"
    println("$mark サンプル ${i+1} ($description):")
    println("  特徴量: Pclass=${sample[0]}, Age=${sample[1]}, male=${sample[2]}")
    println("  予測: ${classNamesMap[prediction]} (期待: ${classNamesMap[expected]})")
    println()
}

個別予測の例:
------------------------------------------------------------
✓ サンプル 1 (1等客室、35歳、女性):
  特徴量: Pclass=1.0, Age=35.0, male=0.0
  予測: 生存 (期待: 生存)

✓ サンプル 2 (3等客室、25歳、男性):
  特徴量: Pclass=3.0, Age=25.0, male=1.0
  予測: 死亡 (期待: 死亡)

✗ サンプル 3 (2等客室、28歳、女性):
  特徴量: Pclass=2.0, Age=28.0, male=0.0
  予測: 死亡 (期待: 生存)



## 12. モデルの保存

In [25]:
import java.io.File

// モデル保存先のパスを構築
val modelDir = when {
    File("model").exists() || File(".").resolve("model").parentFile.exists() -> "model"
    File("../model").parentFile.exists() -> "../model"
    File("app/kotlin/model").parentFile.exists() -> "app/kotlin/model"
    else -> "model" // デフォルト
}

val modelPath = "$modelDir/survived_model.ser"
File(modelPath).parentFile?.mkdirs()

classifier.saveModel(modelPath)

println("=".repeat(60))
println("モデルの保存完了")
println("=".repeat(60))
println("保存先: $modelPath")
println("ファイルサイズ: ${File(modelPath).length()} bytes")

モデルの保存完了
保存先: model/survived_model.ser
ファイルサイズ: 3365 bytes


## まとめ

このノートブックでは、Titanic 生存予測データセットを使った分類問題を TDD で実装しました。

### 実施内容

1. **データの読み込みと探索**: CSV からのデータ読み込み、Pclass ごとの Age 補完、Sex カラムのエンコード
2. **データの可視化**: ヒストグラム、散布図、棒グラフによる分布確認
3. **モデルの訓練**: 決定木モデルの訓練（max_depth=9）
4. **モデルの評価**: 正解率、Precision、Recall、F1 Score の計算
5. **混同行列**: クラス別の予測精度の分析
6. **予測の実行**: 新しいデータでの分類予測
7. **モデルの保存**: モデルの永続化

### 次のステップ

- データを訓練用とテスト用に分割して過学習をチェック
- クロスバリデーションで性能を評価
- クラス不均衡への対応（SMOTE、重み付けなど）
- 特徴量エンジニアリング（Fare、SibSp、Parch など追加）
- Web API 化（Ktor による REST API 実装）

---

お疲れ様でした！Titanic 生存予測モデルの実装が完了しました！